# 02.07 · Debugging agents in LangSmith Studio (first look)

**Module 02 · Tools & Your First Agents** · Part 1 (Core)

| | |
|---|---|
| ⏱️ Estimated time | 60–90 min |
| 📋 Prerequisites | Lessons 02.03 and 02.06. A free LangSmith account (Lesson 00.03). |
| 🔑 Keys | `GROQ_API_KEY` (for the agent), a LangSmith login in the browser; `LANGSMITH_API_KEY` optional |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-12 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Describe** what `langgraph dev` runs (a local Agent Server) and what Studio is (a web UI that connects to it).
2. **Configure** an app with `langgraph.json` (`dependencies`, `graphs`, `env`).
3. **Start** the server from Git Bash, open Studio, run the agent, and step through its nodes and state.
4. **Call** the same server from Python with `langgraph_sdk`.
5. **Fix** the common local problems (CLI not found, port in use, blocking-call errors, Safari/Brave, missing keys).

## 📚 Official reading (verified 2026-10-12)

| Page | Read it for |
|---|---|
| [LangSmith Studio (LangChain)](https://docs.langchain.com/oss/python/langchain/studio) | **Main page.** Set up a LangChain agent for Studio: `langgraph.json`, `.env`, `langgraph dev` |
| [Run a local server](https://docs.langchain.com/oss/python/langgraph/local-server) | Installing `langgraph-cli[inmem]`, server URLs, the Python SDK client, Safari `--tunnel` |
| [Get started with Studio](https://docs.langchain.com/langsmith/quick-start-studio) | The Studio UI: graph view, threads, editing state, forking runs |

---
## Step 0 · Install and set up

`langgraph-cli[inmem]` (needs Python ≥ 3.11) is in the course's `[dev]` extra since Module 02, through the new `studio` extra. If you installed the course with `pip install -e ".[dev]"` **after** pulling Module 02, you have it. Otherwise, in Git Bash with the course environment active, from the repo root:

```bash
python -m pip install -e ".[dev]"      # or just: python -m pip install "langgraph-cli[inmem]>=0.4.33,<1"
```

Use `python -m pip` (not bare `pip`) so it installs into the environment that `python` belongs to: the same rule as the kernel lesson (02.01).

In [ ]:
from importlib import metadata

for package in ("langchain", "langgraph", "langgraph-cli", "langgraph-api", "langgraph-sdk",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Step 1: Find the repository root (Jupyter starts in modules/02-tools-and-agents/).
import sys
from pathlib import Path

REPO_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone.")

# Step 2: Import the course package (installed editable in Lesson 00.02; src/ fallback otherwise).
try:
    import mastery
except ModuleNotFoundError:
    sys.path.insert(0, str(REPO_ROOT / "src"))
    import mastery
    print("ℹ️ Loaded mastery from src/. Fix once: python -m pip install -e \".[dev]\"")
print("Repo root:", REPO_ROOT)

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY', 'LANGSMITH_API_KEY')
# Step 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")

---
## Part 1 · Two pieces: a local server and a web UI

```text
 your machine                                         your browser
┌───────────────────────────────────────────┐        ┌───────────────────────────────┐
│ langgraph dev                              │  HTTP  │ smith.langchain.com/studio     │
│  • imports studio/agent.py:agent           │◀──────▶│  ?baseUrl=http://127.0.0.1:2024│
│  • Agent Server API on 127.0.0.1:2024      │        │  graph view, threads, state    │
│  • in-memory threads/checkpoints           │        └───────────────────────────────┘
│  • calls Groq with YOUR key from .env      │
└───────────────────────────────────────────┘
```

- **`langgraph dev`** runs the **Agent Server** in development mode: it imports your graph, exposes a REST API, keeps threads in memory (they vanish on restart), and **hot-reloads** when you edit the code.
- **Studio** is a page on smith.langchain.com that talks to *your* local server from the browser. It needs you to be **logged in to LangSmith** (free account). The agent's model calls still go from your machine to Groq.
- Traces are uploaded to LangSmith only if tracing is on (`LANGSMITH_TRACING=true` + `LANGSMITH_API_KEY`). Set `LANGSMITH_TRACING=false` to keep run data local.

The app for this lesson is in [`studio/`](studio/): [`langgraph.json`](studio/langgraph.json) and [`agent.py`](studio/agent.py), a finance agent over the sample expense log with the `mastery.tools`.

In [ ]:
# 🧪 Offline: read the app config and check every path it points to.
import json

STUDIO_DIR = REPO_ROOT / "modules" / "02-tools-and-agents" / "studio"
config = json.loads((STUDIO_DIR / "langgraph.json").read_text(encoding="utf-8"))
print(json.dumps(config, indent=2))

for graph_id, spec in config["graphs"].items():
    file_part, _, variable = spec.partition(":")
    path = (STUDIO_DIR / file_part).resolve()
    print(f"graph {graph_id!r}: {path.name} exists={path.exists()} variable={variable!r}")
env_path = (STUDIO_DIR / config["env"]).resolve()
print("env file:", env_path, "exists" if env_path.exists() else "missing (fine if keys are set in your shell)")

**You should now see** the config, `graph 'finance_agent': agent.py exists=True variable='agent'`, and the repo's `.env` path.

| Key | Meaning |
|---|---|
| `dependencies` | What to install when the app is **deployed** (`"."` = this folder). `langgraph dev` uses your current environment instead, which is why `mastery` must be installed there. |
| `graphs` | `graph_id → "file.py:variable"`. The variable is a compiled graph (what `create_agent` returns). |
| `env` | A `.env` file to load (here the repo's), or a dict of variables. |

In [ ]:
# 🧪 Offline: is the CLI installed in THIS environment?
import shutil
import subprocess
import sys
from pathlib import Path

cli = shutil.which("langgraph")
print("python   :", sys.executable)
print("langgraph:", cli or "NOT FOUND on PATH")
if cli:
    same_env = Path(cli).absolute().is_relative_to(Path(sys.prefix).absolute())   # bin/ or Scripts/ of this env
    print("same env :", "✅ yes" if same_env else "⚠️ the CLI is from another environment; use the course env's one")
    print(subprocess.run([cli, "--version"], capture_output=True, text=True, timeout=60).stdout.strip())
else:
    print('Install it: python -m pip install -e ".[dev]"  (or "langgraph-cli[inmem]>=0.4.33,<1")')

**You should now see** `LangGraph CLI, version 0.4.33` (or newer) and `same env : ✅ yes`. On Windows the CLI is `langgraph.exe` in the env's `Scripts` folder, so *same env* checks that the CLI lives inside this kernel's environment (`sys.prefix`). If it says *another environment*, you'd start a server that can't import `mastery`.

> **Why not run the server from this notebook?** `langgraph dev` is a long-running process with its own logs and hot reload. Run it in a terminal you can see and stop with Ctrl+C.

---
## Part 2 · Start the server (Git Bash)

Open a **new Git Bash** window and run:

```bash
cd /c/path/to/langchain-langgraph-mastery          # your repo
conda activate lc-mastery                           # Step 4b's SSL_CERT_FILE fix comes with the env
cd modules/02-tools-and-agents/studio
langgraph dev
```

You should see:

```text
- 🚀 API: http://127.0.0.1:2024
- 🎨 Studio UI: https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024
- 📚 API Docs: http://127.0.0.1:2024/docs
```

and your browser opens the Studio URL (`--no-browser` to stop that). Leave the terminal open. Then come back and run the next cell.

In [ ]:
# 🧪 Offline-safe: is a local server running? (Only talks to 127.0.0.1; prints instructions if not.)
import urllib.error
import urllib.request

SERVER = "http://127.0.0.1:2024"
try:
    with urllib.request.urlopen(f"{SERVER}/info", timeout=3) as response:
        info = json.loads(response.read())
    SERVER_UP = True
    print(f"✅ Agent Server {info.get('version')} (langgraph {info.get('langgraph_py_version')}) at {SERVER}")
    print("Studio:", f"https://smith.langchain.com/studio/?baseUrl={SERVER}")
except (urllib.error.URLError, OSError) as exc:
    SERVER_UP = False
    print(f"⏭️ No server at {SERVER} ({type(exc).__name__}). Start it with `langgraph dev` (Part 2), then rerun this cell.")

---
## Part 3 · A guided tour of Studio

In the Studio tab (log in to LangSmith if asked), pick **`finance_agent`** at the top. Then:

1. **Graph view.** You see `__start__ → model ⇄ tools → __end__`, plus the middleware hooks (`ModelCallLimitMiddleware.before_model` / `.after_model`). Same graph as `agent.get_graph()` in 02.03.
2. **Run it.** In the input panel, add a *human* message: `How much did I spend on groceries in September 2026?` and press **Submit**. Watch the nodes light up as they run.
3. **Read each step.** Click a step in the thread on the right: the `model` step shows the tool call and its arguments; the `tools` step shows the `ToolMessage` with the CSV rows. This is the `updates` stream from 02.06, as a UI.
4. **Inspect state.** Each step shows the full state (`messages`) after it. Count the model calls: is it within the `run_limit=8`?
5. **Fork a step.** Hover a `model` step and **edit** its output (for example, change the tool arguments to `value: "restaurants"`), then re-run from there. Studio creates a **fork**: a new branch of the thread from that checkpoint. You just did time travel (Module 03 explains checkpoints).
6. **New thread.** Start a new thread and ask something out of scope (*"What's the weather in Bogotá?"*). Does the agent say it can't help, as the system prompt asks?
7. **Hot reload.** Edit `SYSTEM_PROMPT` in `studio/agent.py` (say, "answer in Spanish"), save, and watch the terminal: the server reloads. Run again in Studio.
8. **Trace (optional).** If tracing is on, the run also appears in LangSmith under your project, with token counts and latency per step.

> Threads live **in memory**: stopping `langgraph dev` deletes them. That's expected in development.

---
## Part 4 · The same server from Python

Studio is one client of the Agent Server API; `langgraph_sdk` is another (installed with the CLI). This is how a web app would call a deployed agent (Module 08).

In [ ]:
# 💸 Live (needs the server from Part 2 and a GROQ key in .env): stream one run through the API.
import importlib.util

if not SERVER_UP:
    print("⏭️ Skipped: start `langgraph dev` (Part 2) and rerun the server check cell.")
elif not importlib.util.find_spec("langgraph_sdk"):
    print('⏭️ Skipped: langgraph_sdk not installed (python -m pip install -e ".[dev]").')
elif not HAVE_GROQ_KEY:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")
else:
    from langgraph_sdk import get_sync_client

    client = get_sync_client(url=SERVER)
    print("assistants:", [a["graph_id"] for a in client.assistants.search()])
    thread = client.threads.create()
    for chunk in client.runs.stream(thread["thread_id"], "finance_agent", stream_mode="updates",
                                    input={"messages": [{"role": "human", "content": "How much did I spend on groceries in September 2026?"}]}):
        for node, update in (chunk.data or {}).items() if chunk.event == "updates" else []:
            for message in (update or {}).get("messages", []):
                calls = message.get("tool_calls") or []
                print(f"[{node}]", calls[0]["name"] + str(calls[0]["args"]) if calls else str(message.get("content"))[:150])
    print("\nThread id (open it in Studio → Threads):", thread["thread_id"])

**You should now see** `assistants: ['finance_agent']`, then `[model] find_expenses{...}`, `[tools] {"file": "expenses-2026.csv", ...}`, maybe a calculator step, and the answer. Open Studio's thread list: the same thread is there, because Studio and the SDK talk to the same server.

Groceries in September 2026 total **1,694,000 COP** in the sample data. If the agent's number differs, check the steps: did it filter by month as well as category, and did it add with the calculator? This is exactly the kind of mistake Studio makes visible, and what Project 02's dedicated tools fix.

### What just happened?

- `langgraph dev` imported `studio/agent.py`, found `agent` (named `finance_agent` in `langgraph.json`), and exposed it as an **assistant** over HTTP.
- `threads.create()` made a server-side thread; `runs.stream(...)` started a run on it and streamed `updates` back, the same parts you streamed locally in 02.06, as JSON.
- The server, not your notebook, called Groq, with the key it loaded from `.env`. Your notebook only talked to `127.0.0.1`.

## ⚠️ Pitfalls and troubleshooting (Windows / Git Bash first)

| Symptom | Cause | Fix |
|---|---|---|
| `bash: langgraph: command not found` | CLI not installed in the active env, or env not active | `conda activate lc-mastery`, then `python -m pip install -e ".[dev]"`; check with `which langgraph` |
| `ModuleNotFoundError: No module named 'mastery'` in the server log | Server started from a different environment | Same env as the notebook: compare `which python` and `which langgraph` |
| `MissingKeyError: GROQ_API_KEY` at startup | No key in `.env` / shell | Add it to the repo's `.env` (the `env` entry points there) and restart |
| `[Errno 10048]` / *address already in use* | Another server on 2024 | Stop it (Ctrl+C in its terminal) or `langgraph dev --port 2025` and use that port in the Studio URL |
| Windows Firewall prompt | Python opens a local port | *Private networks* is enough; it only listens on 127.0.0.1 |
| Studio: *Failed to load assistants* / can't connect | Browser blocks calls to localhost (Brave Shields, some extensions), or Safari | Allow the site / disable Shields for smith.langchain.com; Safari: `langgraph dev --tunnel` |
| `BlockingError: Blocking call to ...` | A sync I/O call in an async server path | Prefer async code; for learning, `langgraph dev --allow-blocking` |
| `FileNotFoundError` / `CertificateSettingError` about `SSL_CERT_FILE` in the server log | Anaconda's `openssl` activation points `SSL_CERT_FILE` at a file that doesn't exist | Apply the one-time fix from [Lesson 00.02, Step 4b](../00-foundations/00-02-create-course-env.md#step-4b--windows--git-bash-check-and-fix-the-certificate-path) (`conda env config vars set SSL_CERT_FILE=...`), then re-activate the env in **that** terminal |
| Edits to `agent.py` don't apply | Started with `--no-reload`, or a syntax error during reload | Check the terminal; restart `langgraph dev` |
| Threads disappeared | In-memory server restarted | Expected; persistence comes with deployment |

## ⚠️ Production notes

> ⚠️ **Production note: `langgraph dev` is for development only.**
> It's in-memory, single-process and hot-reloading. Deployments run the Agent Server with a persistent backend (Postgres/Redis) via LangSmith Deployment or a self-hosted image (`langgraph build`), covered in Module 08.

> ⚠️ **Production note: what leaves your machine.**
> The Studio page runs in your browser and talks to `127.0.0.1` directly; your prompts go to Groq from the server. With tracing on, inputs and outputs are uploaded to LangSmith. For real user data, set `LANGSMITH_TRACING=false` or configure redaction (Module 04) before you debug with it.

> ⚠️ **Production note: the same graph everywhere.**
> Keep the graph in an importable module (`agent.py:agent`) that notebooks, tests, Studio and the server all import. Don't copy-paste agents between notebooks and apps; that's how a "works in Studio" bug happens.

## 🏋️ Exercises

1. **(Easy)** Ask Studio *"What was my biggest single expense in August 2026?"*. Which tool calls does the agent make, and is the answer right? (Answer key: Arriendo Edificio Nogal, 1,850,000 COP.)
2. **(Medium)** Add `"answer_language": "es"`-style behaviour without changing code: in Studio, open the assistant's **configuration** and see what's editable. Why is the system prompt not there? (Hint: it isn't in `context_schema`.)
3. **(Medium)** Add a second graph to `langgraph.json`: `"calculator_agent": "./calc_agent.py:agent"` with only the calculator tool. Restart and switch between graphs in Studio.
4. **(Stretch)** Make a run hit the `ModelCallLimitMiddleware(run_limit=8)`: ask for something that needs many lookups (*"For every category, list the September total"*). What does the last message say, and how would you raise the limit only for this graph?

<details>
<summary><b>Show solutions</b></summary>

**1.** Typically `find_expenses(column='date', value='2026-08', contains=True)` (20 rows max, so it may miss rows!), then the answer. The 20-row cap is why Project 02 has `top_merchants`: a lookup tool is the wrong tool for "the biggest".

**2.** Studio shows what the graph declares as configurable/context. To make the language selectable, add a `context_schema` with `answer_language` and use it in a `@dynamic_prompt` (02.03, 02.04).

**4.** The last message is `Model call limits exceeded: run limit (8/8)`. Raise `run_limit` in `studio/agent.py` (it only affects this graph).
</details>

## ✅ Checkpoint

- [ ] I can explain the difference between `langgraph dev` (server) and Studio (web UI).
- [ ] I can read `langgraph.json` and say what each key does.
- [ ] I started the server from Git Bash in the course env and opened Studio.
- [ ] I ran the agent, inspected a `model` and a `tools` step, and forked a run.
- [ ] I called the server from Python with `langgraph_sdk`.
- [ ] I know where data goes (local server, Groq, LangSmith when tracing is on).
- [ ] Restart & Run All passes (the server cells skip cleanly when no server is running); outputs cleared before saving.

## 🔎 Research checkpoint

Read the [Module 02 research task](README.md#research-checkpoint): list three things that changed from `create_react_agent` to `create_agent` (Lesson 02.03 Part 5 has a head start). Then, on the [Studio page](https://docs.langchain.com/oss/python/langchain/studio), find whether Studio supports **interrupts/breakpoints** for `create_agent` graphs and how. Log both in [`docs/research-log.md`](../../docs/research-log.md).

## Summary

- `langgraph dev` serves your graph locally (API on 127.0.0.1:2024, in-memory, hot reload); Studio is the browser UI on top.
- `langgraph.json` maps graph ids to `file.py:variable` and points at your `.env`.
- Studio shows every step, the state after it, and lets you fork runs; `langgraph_sdk` calls the same API from code.
- Run the server in the same environment as the course, from a visible terminal.

**Next:** [Project 02: Personal Finance Helper Agent](project/README.md).

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: the **module project**: read [project/README.md](project/README.md) (spec, modes, acceptance criteria).

[Module 02 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)